# Lesson 5 - Build a chatbot that incorporates memory

## Start ollama by docker compose

In [1]:
!docker compose up -d ollama

 Container ollama  Running


## Pull Meta-Llama-3.1-8B-Claude-GGUF model from Hugging Face

In [2]:
!docker compose exec ollama ollama pull hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M

pulling manifest 
pulling e5143516efe0: 100% ▕██████████████████▏ 4.9 GB                         
pulling 783adfd1d253: 100% ▕██████████████████▏  976 B                         
pulling 1a9f0f5ed111: 100% ▕██████████████████▏   22 B                         
pulling d9b87732a16b: 100% ▕██████████████████▏  552 B                         
verifying sha256 digest 
writing manifest ⠋ pulling manifest 
pulling e5143516efe0: 100% ▕██████████████████▏ 4.9 GB                         
pulling 783adfd1d253: 100% ▕██████████████████▏  976 B                         
pulling 1a9f0f5ed111: 100% ▕██████████████████▏   22 B                         
pulling d9b87732a16b: 100% ▕██████████████████▏  552 B                         
verifying sha256 digest 
writing manifest 
success 


## Set Ollama environment variables

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

## Create Ollama Chat Model

In [4]:
from langchain_ollama import ChatOllama

# Initialize the model
model1 = ChatOllama(model="hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M", temperature=0.7, top_k=40)

Let's first use the model directly. ChatOllama are instances of LangChain "BaseChatModel" inheritted from "Runnables", which means they expose a standard interface for interacting with them. To just simply call the model, we can pass in a list of messages to the .invoke method.

In [5]:
from langchain_core.messages import SystemMessage, HumanMessage

output1 = model1.invoke([
    HumanMessage(content="Hi! I'm Bob")
])

The model on its own does not have any concept of state. For example, if you ask a followup question

In [6]:
model1.invoke([HumanMessage(content="What's my name?")])

AIMessage(content='I apologize, but I don\'t have any information about your name. You haven\'t told me what your name is. If you\'d like to share it with me, I\'d be happy to remember it for our conversation. But if not, that\'s perfectly fine too - feel free to just call me "assistant" and we can continue chatting without mentioning names at all. Let me know how else I can assist you!', additional_kwargs={}, response_metadata={'model': 'hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M', 'created_at': '2025-05-26T16:02:37.2957974Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1664022343, 'load_duration': 23015959, 'prompt_eval_count': 15, 'prompt_eval_duration': 8138183, 'eval_count': 86, 'eval_duration': 1631938870, 'model_name': 'hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M'}, id='run--05317cc5-60f6-4fc9-ac99-efe74cacf376-0', usage_metadata={'input_tokens': 15, 'output_tokens': 86, 'total_tokens': 101})

To get around this, we need to pass the entire conversation history into the model. Let's see what happens when we do that:

In [7]:
from langchain_core.messages.ai import AIMessage

model1.invoke(
    [
        HumanMessage(content="Hi! I'm Bob"),
        AIMessage(content="Hello Bob! How can I assist you today?"),
        HumanMessage(content="What's my name?"),
    ]
)

AIMessage(content="Your name is Bob, as you mentioned in your greeting. Is there something specific you'd like to know about or discuss related to the name Bob?", additional_kwargs={}, response_metadata={'model': 'hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M', 'created_at': '2025-05-26T16:02:37.943063824Z', 'done': True, 'done_reason': 'stop', 'total_duration': 630851812, 'load_duration': 18389522, 'prompt_eval_count': 40, 'prompt_eval_duration': 17595468, 'eval_count': 31, 'eval_duration': 593291650, 'model_name': 'hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M'}, id='run--7035ce95-719e-4df8-b09b-99910385ba87-0', usage_metadata={'input_tokens': 40, 'output_tokens': 31, 'total_tokens': 71})

## Memory Persistence

LangGraph implements a built-in persistence layer, making it ideal for chat applications that support multiple conversational turns.

Wrapping our chat model in a minimal LangGraph application allows us to automatically persist the message history, simplifying the development of multi-turn applications.

LangGraph comes with a simple in-memory checkpointer, which we use below. See its documentation for more detail, including how to use different persistence backends (e.g., SQLite or Postgres).

In [8]:
import json
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

# Define a new graph
workflow = StateGraph(state_schema=MessagesState)

# Define the function that calls the model
def call_model(state: MessagesState):
    response = model1.invoke(state["messages"])
    return {"messages": response}

# Define the (single) node in the graph
workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

# Add memory
memory = MemorySaver()

app = workflow.compile(checkpointer=memory)

query = "Hi! I'm Bob."
input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, {"configurable": {"thread_id": "abc123"}})

# only one key "messages" in output dict
output["messages"][-1].pretty_print()  # output contains all messages in state


================================== Ai Message ==================================

Hello Bob, it's nice to meet you! How are you doing today? I'm here if you'd like to chat about anything on your mind or if there are any topics you're curious to discuss. Feel free to ask me questions or just have a friendly conversation.


In [9]:
query = "What's my name?"

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, {"configurable": {"thread_id": "abc123"}})
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Your name is Bob, as you mentioned in your first message. Is there something specific you'd like to know about the name Bob or did you just want to confirm it for yourself? I'm happy discuss names and naming conventions if that interests you!


## Prompt Template

Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [10]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You talk like a pirate. Answer all questions to the best of your ability.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

We can now update our application to incorporate this template:

In [11]:
workflow = StateGraph(state_schema=MessagesState)

def call_model(state: MessagesState):
    prompt = prompt_template.invoke(state)
    response = model1.invoke(prompt)
    return {"messages": response}

workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

We invoke the application in the same way:

In [12]:
config = {"configurable": {"thread_id": "abc345"}}
query = "Hi! I'm Jim."

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()

query = "What is my name?"

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Ahoy there, Jim me bucko! 'Tis pleasure t' make yer acquaintance. Yer lookin' fer some swashbucklin' conversation, eh? Well, ye've come t' right place - ol' Seadog Sam be happy t' spin a yarn or two o'er a bottle o' grog. Whatcha want t' discuss - the high seas, buried treasure, or maybe how t' tie a proper sailors knot? Fire away, matey!
================================== Ai Message ==================================

Thar ye be askin' about yer own name, eh Jimbo? Well, I'll let ye in on a little secret - it's right there in the question yerself provided! Yer name be... wait fer it... JIM! Aye, that be right. Jim be the name o' this here landlubber we're havin' a chat with. Not exactly the most fearsome moniker fer a scallywag on the high seas, but I suppose it'll do fer now. Just don't go expectin' me t' call ye Captain Jim or anythin' like that - that be reserved fer the head buccaneer! Now then, what

Let's now make our prompt a little bit more complicated. Note that we have added a new language input to the prompt. Our application now has two parameters-- the input messages and language. We should update our application's state to reflect this:

In [13]:
from typing import Sequence

from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from typing_extensions import Annotated, TypedDict

prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

class State(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    language: str

workflow = StateGraph(state_schema=State)

def call_model(state: State):
    prompt = prompt_template.invoke(state)
    response = model1.invoke(prompt)
    return {"messages": [response]}

workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "abc456"}}
query = "Hi! I'm Bob."
language = "Spanish"

input_messages = [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages, "language": language},
    config,
)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

¡Hola Bob! Mucho gusto en conocerte. ¿En qué puedo ayudarte hoy?


Note that the entire state is persisted, so we can omit parameters like language if no changes are desired:

In [14]:
query = "What is my name?"

input_messages = [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages, "language": language},
    config
)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Tu nombre es Bob, como mencionaste al inicio de nuestra conversación. ¿Hay algo más que desees saber sobre tu identidad o preferencias? Estoy aquí para ayudarte en lo que necesites.


We invoke the application in the same way:

In [15]:
config = {"configurable": {"thread_id": "abc345"}}
query = "Hi! I'm Jim."

input_messages = [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages, "language": language}, 
    config
)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Hola Jim, encantado de conocerte. ¿En qué puedo ayudarte hoy? Estoy aquí para asistirte en lo que necesites, ya sea responder preguntas, hacer traducciones o proporcionar información sobre una amplia variedad de temas. Por favor, no dudes en hacérmelo saber y haré mi mejor esfuerzo por ayudarte.


In [16]:
query = "What is my name?"

input_messages = [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages, "language": language}, 
    config
)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Tu nombre es Jim. ¿Hay algo más en lo que pueda asistirte hoy?


## Managing Conversation History

One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

Importantly, you will want to do this BEFORE the prompt template but AFTER you load previous messages from Message History.

We can do this by adding a simple step in front of the prompt that modifies the messages key appropriately, and then wrap that new chain in the Message History class.

LangChain comes with a few built-in helpers for managing a list of messages. In this case we'll use the trim_messages helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages:

In [17]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=65,
    strategy="last",
    token_counter=model1,
    include_system=True,
    allow_partial=False,
    start_on="human",
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="hi! I'm bob", additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

To use it in our chain, we just need to run the trimmer before we pass the messages input to our prompt.

In [18]:
workflow = StateGraph(state_schema=State)

def call_model(state: State):
    trimmed_messages = trimmer.invoke(state["messages"])
    prompt = prompt_template.invoke(
        {"messages": trimmed_messages, "language": state["language"]}
    )
    response = model1.invoke(prompt)
    return {"messages": [response]}

workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

Now if we try asking the model our name, it won't know it since we trimmed that part of the chat history:

In [19]:
config = {"configurable": {"thread_id": "abc567"}}
query = "What is my name?"
language = "English"

input_messages = messages + [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages, "language": language},
    config,
)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

Your name is Bob, as you mentioned earlier.


But if we ask about information that is within the last few messages, it remembers:

In [20]:
config = {"configurable": {"thread_id": "abc678"}}
query = "What math problem did I ask?"
language = "English"

input_messages = messages + [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages, "language": language},
    config,
)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

You asked the math problem "whats 2 + 2".


## Streaming

Now we've got a functioning chatbot. However, one really important UX consideration for chatbot applications is streaming. LLMs can sometimes take a while to respond, and so in order to improve the user experience one thing that most applications do is stream back each token as it is generated. This allows the user to see progress.

It's actually super easy to do this!

By default, .stream in our LangGraph application streams application steps-- in this case, the single step of the model response. Setting stream_mode="messages" allows us to stream output tokens instead:

In [21]:
config = {"configurable": {"thread_id": "abc789"}}
query = "Hi I'm Todd, please tell me a joke."
language = "English"

input_messages = [HumanMessage(query)]
for chunk, metadata in app.stream({"messages": input_messages, "language": language}, config, stream_mode="messages"):
    if isinstance(chunk, AIMessage):  # Filter to just model responses
        print(chunk.content, end="|")

Sure| thing|,| Todd|!| Here|'s| a| silly| joke| for| you|:

|What| do| you| call| a| bear| with| no| teeth|?| 
|A| g|ummy| bear|!

|I| hope| that| joke| made| you| chuck|le|.| Let| me| know| if| you|'d| like| to| hear| any| other| jokes| -| I|'ve| got| plenty| more| where| that| came| from|!||